# Weekly Metrics

Weekly survey-progress and engineering-health summary for the Rubin/LSST
community forum. Reports the trailing `n_days` (default 7) plus season-to-date
totals since the start of steady-state LSST operations, and auto-fills a draft
community post with the headline numbers.

This is the recurring follow-up to the kickoff post
[2026-07-10 Early Operations Update: Start of LSST](https://community.lsst.org/t/2026-07-10-early-operations-update-start-of-lsst/12280).

In [ ]:
# Parameters cell. Set defaults here (overridden by Times Square at runtime).
import rubin_nights.dayobs_utils as rn_dayobs

# Last day_obs of the reporting window (defaults to today).
day_obs_max = rn_dayobs.day_obs_str_to_int(rn_dayobs.today_day_obs())
# Length of the weekly reporting window, in nights.
n_days = 7
# Start of steady-state LSST operations: the 10-year LSST began the night of
# 29 June 2026. Season-to-date totals and the "[N]th week" count run from here.
season_start_day_obs = 20260629

In [ ]:
# Keep rubin_nights current when running under Times Square (matches the other
# notebooks in this repo). Set not_times_square = True to skip when developing.
try:
    not_times_square
except NameError:
    not_times_square = False

if not not_times_square:
    print("updating rubin_nights")
    !pip install --user --upgrade git+https://github.com/lsst-sims/rubin_nights.git --no-deps > /dev/null 2>&1

In [ ]:
# Configuration for password/RSP token files (copied from efficiency.ipynb).
import getpass
import os

# Who is running the notebook?
username = getpass.getuser()
# Where is the notebook running? (RSPs are 'special')
current_location = os.getenv("EXTERNAL_INSTANCE_URL", "")

# RUBIN_SIM_DATA_DIR at usdf
if "usdf" in current_location:
    os.environ["RUBIN_SIM_DATA_DIR"] = "/sdf/data/rubin/shared/rubin_sim_data"

# TOKEN CONFIGURATION
if current_location != "":
    # You are on an RSP - use the default RSP values (summit/base/USDF).
    tokenfile = None
    site = None
else:
    # Outside an RSP: use USDF with your own USDF-RSP token.
    # See https://rsp.lsst.io/guides/auth/creating-user-tokens.html
    tokenfile = None
    site = None

In [ ]:
# Imports
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from astropy.time import Time, TimeDelta

from rubin_nights import connections
import rubin_nights.dayobs_utils as rn_dayobs
import rubin_nights.augment_visits as augment_visits
import rubin_nights.observatory_status as observatory_status
import rubin_nights.rubin_scheduler_addons as rn_sch

# Science program list (BLOCK-* survey programs) with a hard-coded fallback,
# exactly as efficiency.ipynb does it.
try:
    from rubin_nights.reference_values import SCIENCE_PROGRAMS

    programs = list(SCIENCE_PROGRAMS)
except ImportError:
    programs = [
        "BLOCK-365",
        "BLOCK-407",
        "BLOCK-408",
        "BLOCK-416",
        "BLOCK-417",
        "BLOCK-419",
        "BLOCK-421",
    ]

# Band plotting colors (fall back to a plain dict off-RSP).
try:
    from lsst.utils.plotting import get_multiband_plot_colors

    band_colors = get_multiband_plot_colors()
except Exception:
    band_colors = {
        "u": "#56b4e9",
        "g": "#008060",
        "r": "#ff4000",
        "i": "#850000",
        "z": "#6600cc",
        "y": "#000000",
    }
bands_ordered = ["u", "g", "r", "i", "z", "y"]

# PSF FWHM to psf_sigma conversion (from image_quality_trending.ipynb).
SIG2FWHM = 2.0 * np.sqrt(2.0 * np.log(2.0))
PIXEL_SCALE = 0.2  # arcsec / pixel

# Delivered image-quality design target by band (arcsec, LSST SRD median seeing).
# Reference values; override here if a project spec supersedes them.
DESIGN_FWHM = {
    "u": 0.92,
    "g": 0.87,
    "r": 0.83,
    "i": 0.80,
    "z": 0.78,
    "y": 0.76,
}

logging.getLogger("rubin_nights").setLevel(logging.INFO)
# --- additions for sky-coverage, DDF/WFD, and prompt-processing metrics ---
import healpy as hp

try:
    import skyproj

    _have_skyproj = True
except Exception:
    _have_skyproj = False

from rubin_nights.influx_query import InfluxQueryClient

try:
    from rubin_sim.data import get_baseline

    _have_baseline = True
except Exception:
    _have_baseline = False

try:
    from rubin_scheduler.utils import ddf_locations, angular_separation
except Exception:
    ddf_locations = None

In [ ]:
# Connect to data services (consdb, EFD, exposure log, narrative log).
endpoints = connections.get_clients(tokenfile=tokenfile, site=site)

In [ ]:
# Build the two reporting windows: the trailing week and season-to-date.
def day_obs_range(day_obs_min, day_obs_max):
    """Inclusive list of integer day_obs between two integer day_obs."""
    one_day = TimeDelta(1, format="jd")
    span = (
        rn_dayobs.day_obs_to_time(day_obs_max) - rn_dayobs.day_obs_to_time(day_obs_min)
    ).jd
    days = rn_dayobs.day_obs_to_time(day_obs_min) + one_day * np.arange(0, span + 1)
    return [rn_dayobs.day_obs_str_to_int(rn_dayobs.time_to_day_obs(d)) for d in days]


# Week window: n_days nights up to and including day_obs_max.
week_min = rn_dayobs.day_obs_str_to_int(
    rn_dayobs.time_to_day_obs(
        rn_dayobs.day_obs_to_time(day_obs_max) - TimeDelta(n_days - 1, format="jd")
    )
)
week_max = day_obs_max

# Season window: from steady-state start through day_obs_max.
season_min = season_start_day_obs
season_max = day_obs_max

week_day_obs = day_obs_range(week_min, week_max)
season_day_obs = day_obs_range(season_min, season_max)

n_week = int(np.ceil(len(season_day_obs) / 7))  # steady-state week number

print(f"Week window:   {week_min} to {week_max} ({len(week_day_obs)} nights)")
print(f"Season window: {season_min} to {season_max} ({len(season_day_obs)} nights)")
print(f"Steady-state week number: {n_week}")

In [ ]:
# Fetch visits for the full season window (the week is a subset), augment them,
# and flag bad / close-loop visits. We fetch all programs so slew/idle math sees
# the whole night, then select science programs where needed.
def fetch_visits(day_obs_min, day_obs_max):
    query = (
        "select v.*, q.* "
        "from cdb_lsstcam.visit1 as v "
        "left join cdb_lsstcam.visit1_quicklook as q on v.visit_id = q.visit_id "
        f"where v.day_obs >= {day_obs_min} and v.day_obs <= {day_obs_max} "
        "and v.img_type not in ('bias', 'flat', 'dark')"
    )
    visits = endpoints["consdb"].query(query)
    if len(visits) == 0:
        return visits
    visits = augment_visits.augment_visits(visits, "lsstcam")
    visits = visits.reset_index(drop=True)
    return visits


visits = fetch_visits(season_min, season_max)
print(f"Fetched {len(visits)} visits over the season window.")

if len(visits) > 0:
    # Flag known-bad visits (lsst-dm excluded list + exposure-log junk).
    visits["bad_flag"] = 0
    bad_visit_ids = augment_visits.fetch_excluded_visits("lsstcam")
    idx = visits.query("visit_id in @bad_visit_ids").index
    visits.loc[idx, "bad_flag"] = 1

    ee = endpoints["exposure_log"].query_log(
        rn_dayobs.day_obs_to_time(season_min), rn_dayobs.day_obs_to_time(season_max)
    )
    if len(ee) > 0:

        def make_visit_id(x):
            return f"{x.day_obs:d}{x.seq_num:05d}"

        junk = ee.query("exposure_flag == 'junk'").apply(make_visit_id, axis=1).values
        if len(junk) > 0:
            idx = visits.query("visit_id in @junk").index
            visits.loc[idx, "bad_flag"] = 1

    # Flag close_loop visits as bad for efficiency/IQ purposes.
    idx = visits.query("observation_reason.str.contains('close_loop')").index
    visits.loc[idx, "bad_flag"] = 1

    # Convenience masks / derived columns.
    visits["is_science"] = visits["science_program"].isin(programs)
    visits["is_week"] = visits["day_obs"].between(week_min, week_max)
    if "psf_sigma_median" in visits.columns:
        visits["psf_fwhm"] = visits["psf_sigma_median"] * SIG2FWHM * PIXEL_SCALE
    else:
        visits["psf_fwhm"] = np.nan

In [ ]:
# Dome open/close and night hours per night (drives open-shutter fraction).
dome_open = observatory_status.get_dome_open_close(
    rn_dayobs.day_obs_to_time(season_min),
    rn_dayobs.day_obs_to_time(season_max) + TimeDelta(1, format="jd"),
    endpoints["efd"],
)


def night_hours_table(day_obs_list):
    rows = []
    for day_obs in day_obs_list:
        dome_day = dome_open.query("day_obs == @day_obs")
        if (len(dome_day) == 0) or ("sunset12" not in dome_day.columns):
            sunset, sunrise = rn_dayobs.day_obs_sunset_sunrise(day_obs, -12)
            night_hours = (sunrise.mjd - sunset.mjd) * 24
            open_hours = 0.0
        else:
            night_hours = dome_day.night_hours.iloc[0]
            open_hours = dome_day.open_hours.sum()
        rows.append(
            pd.Series(
                {"night_hours": night_hours, "open_hours": open_hours}, name=day_obs
            )
        )
    return pd.DataFrame(rows)


night_times = night_hours_table(season_day_obs)
night_times["is_week"] = night_times.index.to_series().between(week_min, week_max)

## 1. Open-shutter time & science visits

Open-shutter (on-sky integration) hours and the number of science-ready visits,
for the trailing week and season-to-date.

In [ ]:
def open_shutter_summary(sci):
    """Open-shutter hours and science visit count for a science-visit frame."""
    good = sci.query("bad_flag == 0")
    return pd.Series(
        {
            "science_visits": len(good),
            "open_shutter_hours": good["exp_time"].sum() / 3600.0,
        }
    )


sci_all = visits.query("is_science")
m1_week = open_shutter_summary(sci_all.query("is_week"))
m1_season = open_shutter_summary(sci_all)

m1 = pd.DataFrame({"week": m1_week, "season_to_date": m1_season})
display(m1.round(1))

## 2. Visits by band

ugrizy breakdown of science visits.

In [ ]:
def band_counts(sci):
    good = sci.query("bad_flag == 0")
    return good["band"].value_counts().reindex(bands_ordered, fill_value=0)


m2_week = band_counts(sci_all.query("is_week"))
m2_season = band_counts(sci_all)
m2 = pd.DataFrame({"week": m2_week, "season_to_date": m2_season})
display(m2)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(bands_ordered))
ax.bar(
    x,
    m2["week"].values,
    color=[band_colors.get(b, "gray") for b in bands_ordered],
)
ax.set_xticks(x)
ax.set_xticklabels(bands_ordered)
ax.set_xlabel("Band")
ax.set_ylabel("Science visits (this week)")
ax.set_title(f"Visits by band, {week_min}-{week_max}")
ax.grid(alpha=0.3, axis="y")
plt.show()

## 3. Delivered image quality (PSF FWHM) vs. design, by band

Median delivered PSF FWHM per band (from `psf_sigma_median`), compared to the
per-band design target. Uses good science visits in the trailing week.

In [ ]:
iq = sci_all.query("is_week and bad_flag == 0 and psf_fwhm == psf_fwhm")
m3 = pd.DataFrame(index=bands_ordered)
m3["median_fwhm"] = iq.groupby("band")["psf_fwhm"].median().reindex(bands_ordered)
m3["design_fwhm"] = [DESIGN_FWHM.get(b, np.nan) for b in bands_ordered]
m3["delta"] = m3["median_fwhm"] - m3["design_fwhm"]
display(m3.round(3))

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(bands_ordered))
ax.bar(
    x,
    m3["median_fwhm"].values,
    color=[band_colors.get(b, "gray") for b in bands_ordered],
    label="delivered (median)",
)
ax.plot(x, m3["design_fwhm"].values, "kD", label="design target")
ax.set_xticks(x)
ax.set_xticklabels(bands_ordered)
ax.set_xlabel("Band")
ax.set_ylabel("PSF FWHM (arcsec)")
ax.set_title(f"Median delivered IQ vs. design, {week_min}-{week_max}")
ax.legend()
ax.grid(alpha=0.3, axis="y")
plt.show()

# Overall median across all bands (used in the draft post).
median_fwhm_week = iq["psf_fwhm"].median()
# Weighted design target for the same band mix, for the on/above/below call.
_band_n = iq["band"].value_counts()
_design_weighted = (
    np.average(
        [DESIGN_FWHM.get(b, np.nan) for b in _band_n.index],
        weights=_band_n.values,
    )
    if len(_band_n)
    else np.nan
)
print(f"Overall median delivered FWHM this week: {median_fwhm_week:.2f} arcsec")
print(f"Band-mix-weighted design target: {_design_weighted:.2f} arcsec")

## 4. Fault & weather downtime

Hours lost to technical faults vs. weather, from the narrative log (night-time
values reported by observing specialists).

In [ ]:
def downtime_hours(day_obs_min, day_obs_max):
    logs = endpoints["narrative_log"].query_log(
        rn_dayobs.day_obs_to_time(day_obs_min),
        rn_dayobs.day_obs_to_time(day_obs_max),
        {"min_time_lost": "0.00001"},
    )
    fault_h = weather_h = 0.0
    if len(logs) > 0:
        logs = logs.query("not component.str.contains('AuxTel')")
        fault_h = logs.query("time_lost_type == 'fault'")["time_lost"].sum()
        weather_h = logs.query("time_lost_type == 'weather'")["time_lost"].sum()
    return pd.Series({"fault_hours": fault_h, "weather_hours": weather_h})


m4_week = downtime_hours(week_min, week_max)
m4_season = downtime_hours(season_min, season_max)
m4 = pd.DataFrame({"week": m4_week, "season_to_date": m4_season})
display(m4.round(2))

## 5. Open-shutter efficiency

Fraction of the night spent in open-shutter integration, and the slew/settle
efficiency relative to the ideal scheduler model. Uses the model slew times
from `rubin_nights`, following `efficiency.ipynb`.

In [ ]:
wait_before_slew = 1.6
settle = 1.5
max_scatter = 10

m5 = pd.Series(dtype=float)
if len(sci_all.query("is_week")) > 0:
    week_visits = visits.query("is_week").sort_values("visit_id").reset_index(drop=True)
    week_visits, _ = rn_sch.add_model_slew_times(
        week_visits,
        endpoints["efd"],
        model_settle=wait_before_slew + settle,
        dome_crawl=True,
        slew_while_changing_filter=False,
    )
    valid_overhead = np.min(
        [
            np.where(
                np.isnan(week_visits.slew_model.values),
                0,
                week_visits.slew_model.values,
            )
            + max_scatter,
            week_visits.visit_gap.values,
        ],
        axis=0,
    )
    week_visits["overhead"] = valid_overhead

    g = week_visits.query("is_science and bad_flag == 0")
    slew_eff = g.exp_time.sum() / (g.dark_time.sum() + g.overhead.sum())
    ideal_eff = g.exp_time.sum() / (g.dark_time.sum() + g.slew_model_ideal.sum())

    # Open-shutter fraction of available night hours (week).
    week_night_hours = night_times.query("is_week").night_hours.sum()
    open_shutter_fraction = (
        g.exp_time.sum() / (week_night_hours * 3600.0)
        if week_night_hours > 0
        else np.nan
    )

    m5 = pd.Series(
        {
            "open_shutter_efficiency": slew_eff,
            "ideal_model_efficiency": ideal_eff,
            "slew_ratio": slew_eff / ideal_eff if ideal_eff else np.nan,
            "open_shutter_fraction_of_night": open_shutter_fraction,
        }
    )
    display(m5.round(3).to_frame("week"))
else:
    print("No science visits in the week window; skipping efficiency metric.")

## 6. DDF vs. WFD allocation

Split of science time between Deep Drilling Fields (DDF) and the Wide-Fast-Deep
(WFD) main survey. DDF visits are tagged by a `ddf_*` token in `target_name`
(e.g. `ddf_elaiss1`, `ddf_xmm_lss`). Early in the survey the DDF cadence may be
barely active, in which case this degrades to a "not yet significantly active"
note.

In [ ]:
# Tag DDF vs WFD on the already-fetched visits.
# DDF visits carry a "ddf_*" token in target_name (scheduler footprint label).
if len(visits) > 0:
    _tname = visits["target_name"].fillna("").astype(str)
    visits["is_ddf"] = _tname.str.contains("ddf", case=False)
    visits["is_wfd"] = visits["is_science"] & ~visits["is_ddf"]


def alloc_summary(df):
    good = df.query("bad_flag == 0")
    ddf = good.query("is_ddf")
    wfd = good.query("is_wfd")
    return pd.Series(
        {
            "ddf_visits": len(ddf),
            "wfd_visits": len(wfd),
            "ddf_open_shutter_hours": ddf["exp_time"].sum() / 3600.0,
            "wfd_open_shutter_hours": wfd["exp_time"].sum() / 3600.0,
        }
    )


sci_science = visits.query("is_science")  # DDF + WFD (all science programs)
m6_week = alloc_summary(sci_science.query("is_week"))
m6_season = alloc_summary(sci_science)
m6 = pd.DataFrame({"week": m6_week, "season_to_date": m6_season})
display(m6.round(1))

ddf_active = m6_week["ddf_visits"] >= 50  # threshold for "meaningfully active"
if not ddf_active:
    print(
        f"DDF cadence not yet significantly active "
        f"({int(m6_week['ddf_visits'])} DDF visits this week)."
    )

## 7. Sky coverage & depth vs. baseline

Season-to-date cumulative sky area imaged to a set of coadded depth thresholds,
per band, using the per-visit 5σ limiting magnitude (`stats_mag_lim_median`) as
the single-visit m5. Coadded depth is binned onto a HEALPix grid
(nside=64) and compared to the 10-year baseline opsim (`get_baseline()`,
the PSTN-057-style reference) for the equivalent elapsed fraction of the survey.

In [ ]:
NSIDE = 64
PIX_AREA = hp.nside2pixarea(NSIDE, degrees=True)  # deg^2 per pixel
DEPTH_THRESHOLDS = [24.0, 24.5, 25.0]


def coadd_m5(m5_values):
    """Coadded 5-sigma depth from a set of single-visit m5 values."""
    m5_values = np.asarray(m5_values, dtype=float)
    m5_values = m5_values[np.isfinite(m5_values)]
    if len(m5_values) == 0:
        return np.nan
    return 1.25 * np.log10(np.sum(10.0 ** (0.8 * m5_values)))


def coverage_by_band(df):
    """Per-band {threshold: area_deg2} from a visits frame with m5 + coords."""
    out = {}
    good = df.query("bad_flag == 0").dropna(
        subset=["stats_mag_lim_median", "s_ra", "s_dec"]
    )
    for band in bands_ordered:
        b = good[good["band"] == band]
        if len(b) == 0:
            out[band] = {t: 0.0 for t in DEPTH_THRESHOLDS}
            continue
        pix = hp.ang2pix(NSIDE, b["s_ra"].values, b["s_dec"].values, lonlat=True)
        coadd = (
            pd.Series(b["stats_mag_lim_median"].values)
            .groupby(pix)
            .apply(lambda s: coadd_m5(s.values))
        )
        out[band] = {t: float((coadd > t).sum()) * PIX_AREA for t in DEPTH_THRESHOLDS}
    return out


# Season-to-date coverage (science visits).
cov_season = coverage_by_band(sci_science)
m7 = pd.DataFrame({band: cov_season[band] for band in bands_ordered}).T
m7.columns = [f"area_deg2_m5>{t}" for t in DEPTH_THRESHOLDS]
m7.index.name = "band"
print("Season-to-date sky area (deg^2) above coadded-depth thresholds:")
display(m7.round(0))

# Headline number for the post: r-band area to m5 > 24.5.
sky_area_r_245 = cov_season.get("r", {}).get(24.5, np.nan)
print(f"\nr-band area to m5 > 24.5: {sky_area_r_245:.0f} deg^2 (season-to-date)")

In [ ]:
# Baseline comparison: what area does the 10yr baseline reach over the same
# elapsed fraction of the survey? Report survey / baseline as a single ratio.
baseline_ratio = np.nan
if _have_baseline:
    try:
        import sqlite3

        baseline_db = get_baseline()
        con = sqlite3.connect(baseline_db)
        # Baseline observations use MJD; take the same number of nights elapsed
        # from the sim start as our season window.
        n_elapsed = len(season_day_obs)
        obs = pd.read_sql_query(
            "select observationStartMJD, fieldRA, fieldDec, filter, fiveSigmaDepth "
            "from observations",
            con,
        )
        con.close()
        mjd0 = obs["observationStartMJD"].min()
        obs_slice = obs[obs["observationStartMJD"] <= mjd0 + n_elapsed]
        r = obs_slice[obs_slice["filter"] == "r"].dropna(
            subset=["fiveSigmaDepth", "fieldRA", "fieldDec"]
        )
        if len(r) > 0:
            pix = hp.ang2pix(
                NSIDE, r["fieldRA"].values, r["fieldDec"].values, lonlat=True
            )
            coadd = (
                pd.Series(r["fiveSigmaDepth"].values)
                .groupby(pix)
                .apply(lambda s: coadd_m5(s.values))
            )
            baseline_area_r_245 = float((coadd > 24.5).sum()) * PIX_AREA
            if baseline_area_r_245 > 0:
                baseline_ratio = sky_area_r_245 / baseline_area_r_245
            print(
                f"Baseline r-band area to m5 > 24.5 over first {n_elapsed} nights: "
                f"{baseline_area_r_245:.0f} deg^2"
            )
            print(f"Survey / baseline area ratio: {baseline_ratio:.2f}")
    except Exception as e:
        print(f"Baseline comparison unavailable: {type(e).__name__}: {e}")
else:
    print("rubin_sim baseline not available; skipping baseline ratio.")

In [ ]:
# Sky map of season-to-date r-band coadded depth.
good_r = sci_science.query("bad_flag == 0 and band == 'r'").dropna(
    subset=["stats_mag_lim_median", "s_ra", "s_dec"]
)
if len(good_r) > 0:
    hpmap = np.full(hp.nside2npix(NSIDE), np.nan)
    pix = hp.ang2pix(NSIDE, good_r["s_ra"].values, good_r["s_dec"].values, lonlat=True)
    coadd = (
        pd.Series(good_r["stats_mag_lim_median"].values)
        .groupby(pix)
        .apply(lambda s: coadd_m5(s.values))
    )
    hpmap[coadd.index.values] = coadd.values

    if _have_skyproj:
        fig, ax = plt.subplots(figsize=(10, 5))
        sp = skyproj.McBrydeSkyproj(ax=ax)
        sp.draw_hpxmap(hpmap, zoom=False)
        ax.set_title("Season-to-date r-band coadded depth (m5)")
        plt.show()
    else:
        hp.mollview(hpmap, title="Season-to-date r-band coadded depth (m5)", unit="mag")
        plt.show()
else:
    print("No r-band science visits with depth for a sky map.")

## 8. Prompt-processing throughput (DIA sources & SS objects)

Difference-image (DIA) source counts and Solar System object associations from
prompt processing, via the `lsst.prompt` metrics database. **These are
difference-image detections and ephemeris associations — not alert-broker
"alerts issued" counts, and not deduplicated distinct objects.** Prompt
processing also covers only a subset of science visits, so the processed-visit
coverage fraction is reported alongside.

In [ ]:
def prompt_throughput(t_start, t_end):
    """Total good DIA sources, SS associations, and processed-visit count."""
    out = {
        "dia_sources_good": np.nan,
        "dia_sources_all": np.nan,
        "ss_object_assoc": np.nan,
        "processed_visits": np.nan,
    }
    try:
        pc = InfluxQueryClient("usdf-dev", db_name="lsst.prompt")
        dia = pc.select_time_series(
            "lsst.prompt.prod.numDiaSourcesGood",
            ["visit", "detector", "numAllDiaSources", "numGoodDiaSources", "run"],
            t_start,
            t_end,
        )
        if len(dia) > 0 and "run" in dia:
            dia = dia[dia["run"].str.startswith("LSSTCam/")]
            out["dia_sources_good"] = float(dia["numGoodDiaSources"].sum())
            out["dia_sources_all"] = float(dia["numAllDiaSources"].sum())
            out["processed_visits"] = int(dia["visit"].astype(int).nunique())
        ss = pc.select_time_series(
            "lsst.prompt.prod.numSsObjects",
            ["visit", "detector", "NumSsObjectsMetric", "run"],
            t_start,
            t_end,
        )
        if len(ss) > 0 and "run" in ss:
            ss = ss[ss["run"].str.startswith("LSSTCam/")]
            out["ss_object_assoc"] = float(ss["NumSsObjectsMetric"].sum())
    except Exception as e:
        print(f"Prompt-processing metrics unavailable: {type(e).__name__}: {e}")
    return pd.Series(out)


# Influx wants a slightly padded end time to catch the last night's morning.
_week_t0 = rn_dayobs.day_obs_to_time(week_min)
_week_t1 = rn_dayobs.day_obs_to_time(week_max) + TimeDelta(1, format="jd")
_season_t0 = rn_dayobs.day_obs_to_time(season_min)
_season_t1 = rn_dayobs.day_obs_to_time(season_max) + TimeDelta(1, format="jd")

m8_week = prompt_throughput(_week_t0, _week_t1)
m8_season = prompt_throughput(_season_t0, _season_t1)
m8 = pd.DataFrame({"week": m8_week, "season_to_date": m8_season})

# Processed-visit coverage relative to science visits.
m8.loc["science_visits", "week"] = m1_week["science_visits"]
m8.loc["science_visits", "season_to_date"] = m1_season["science_visits"]
m8.loc["processed_coverage", "week"] = (
    m8_week["processed_visits"] / m1_week["science_visits"]
    if m1_week["science_visits"]
    else np.nan
)
m8.loc["processed_coverage", "season_to_date"] = (
    m8_season["processed_visits"] / m1_season["science_visits"]
    if m1_season["science_visits"]
    else np.nan
)
display(m8)

In [ ]:
# Good DIA sources per night (week window).
try:
    pc = InfluxQueryClient("usdf-dev", db_name="lsst.prompt")
    dia = pc.select_time_series(
        "lsst.prompt.prod.numDiaSourcesGood",
        ["visit", "detector", "numGoodDiaSources", "run"],
        _week_t0,
        _week_t1,
    )
    if len(dia) > 0 and "run" in dia:
        dia = dia[dia["run"].str.startswith("LSSTCam/")].copy()
        dia["visit"] = dia["visit"].astype(int)
        dia["day_obs"] = dia["visit"] // 100000
        nightly = dia.groupby("day_obs")["numGoodDiaSources"].sum()
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(np.arange(len(nightly)), nightly.values, color="C0")
        ax.set_xticks(np.arange(len(nightly)))
        ax.set_xticklabels(nightly.index, rotation=90)
        ax.set_ylabel("Good DIA sources")
        ax.set_title(f"Good DIA sources per night, {week_min}-{week_max}")
        ax.grid(alpha=0.3, axis="y")
        plt.show()
    else:
        print("No prompt-processing DIA data for the week window.")
except Exception as e:
    print(f"Prompt-processing time series unavailable: {type(e).__name__}: {e}")

## Summary

Headline numbers for the week and season-to-date.

In [ ]:
def _num(x):
    try:
        return float(x)
    except Exception:
        return np.nan


summary = pd.DataFrame(
    {
        "week": {
            "science_visits": m1_week["science_visits"],
            "open_shutter_hours": round(m1_week["open_shutter_hours"], 1),
            "median_fwhm_arcsec": round(float(median_fwhm_week), 2),
            "fault_hours": round(float(m4_week["fault_hours"]), 1),
            "weather_hours": round(float(m4_week["weather_hours"]), 1),
            "open_shutter_efficiency": round(
                float(m5.get("open_shutter_efficiency", np.nan)), 3
            ),
            "ddf_visits": int(m6_week["ddf_visits"]),
            "wfd_visits": int(m6_week["wfd_visits"]),
            "sky_area_r_m5>24.5_deg2": round(_num(sky_area_r_245), 0),
            "dia_sources_good": _num(m8_week["dia_sources_good"]),
            "ss_object_assoc": _num(m8_week["ss_object_assoc"]),
            "prompt_processed_visits": _num(m8_week["processed_visits"]),
        },
        "season_to_date": {
            "science_visits": m1_season["science_visits"],
            "open_shutter_hours": round(m1_season["open_shutter_hours"], 1),
            "median_fwhm_arcsec": np.nan,
            "fault_hours": round(float(m4_season["fault_hours"]), 1),
            "weather_hours": round(float(m4_season["weather_hours"]), 1),
            "open_shutter_efficiency": np.nan,
            "ddf_visits": int(m6_season["ddf_visits"]),
            "wfd_visits": int(m6_season["wfd_visits"]),
            "sky_area_r_m5>24.5_deg2": round(_num(sky_area_r_245), 0),
            "dia_sources_good": _num(m8_season["dia_sources_good"]),
            "ss_object_assoc": _num(m8_season["ss_object_assoc"]),
            "prompt_processed_visits": _num(m8_season["processed_visits"]),
        },
    }
)
display(summary)

## Draft community post

Auto-filled with the week's numbers. Numeric slots are computed; the bracketed
`[...]` prose is left for you to complete before posting. Tone/framing reference:
the [kickoff post](https://community.lsst.org/t/2026-07-10-early-operations-update-start-of-lsst/12280) --
keep it professional but accessible, and link technical notes (RTN-122,
SOTN-004, PSTN-057) rather than dumping tables.

In [ ]:
def ordinal(n):
    if 10 <= n % 100 <= 20:
        suffix = "th"
    else:
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix}"


x_hours = m1_week["open_shutter_hours"]
y_visits = int(m1_week["science_visits"])
z_fwhm = float(median_fwhm_week)

if np.isnan(_design_weighted) or np.isnan(z_fwhm):
    vs_design = "[on/above/below]"
elif z_fwhm <= _design_weighted * 0.98:
    vs_design = "better than"
elif z_fwhm >= _design_weighted * 1.02:
    vs_design = "above"
else:
    vs_design = "on"

# Footprint sentence from Metric 7 (season-to-date r-band area to m5 > 24.5).
if np.isfinite(sky_area_r_245) and sky_area_r_245 > 0:
    footprint_sentence = (
        f"Season-to-date, roughly {sky_area_r_245:,.0f} deg\u00b2 has been imaged "
        f"to a coadded r-band depth beyond 24.5 mag"
    )
    if np.isfinite(baseline_ratio):
        footprint_sentence += (
            f", about {baseline_ratio*100:.0f}% of the PSTN-057 baseline "
            f"expectation for this point in the survey"
        )
    footprint_sentence += "."
else:
    footprint_sentence = "[1 sentence on survey footprint/depth progress vs. PSTN-057.]"

# DDF sentence from Metric 6 (only if meaningfully active).
if m6_week["ddf_visits"] >= 50:
    ddf_sentence = (
        f" Deep Drilling Fields received {int(m6_week['ddf_visits'])} visits this "
        f"week ({int(m6_season['ddf_visits'])} season-to-date)."
    )
else:
    ddf_sentence = " Deep Drilling Field cadence is not yet significantly active."

# Prompt-processing sentence from Metric 8 (caveated: detections, not alerts).
dia_good = m8_week["dia_sources_good"]
proc_visits = m8_week["processed_visits"]
if np.isfinite(dia_good) and np.isfinite(proc_visits) and proc_visits > 0:
    prompt_sentence = (
        f"Prompt processing detected {dia_good/1e6:.1f} million difference-image "
        f"(DIA) sources across {int(proc_visits)} processed visits this week "
        f"(these are image-difference detections, not deduplicated objects or "
        f"issued alerts)."
    )
else:
    prompt_sentence = "[Prompt-processing throughput unavailable this week.]"

post = f"""Following the completion of Early Operations, Rubin Observatory is now in its {ordinal(n_week)} week of steady-state LSST survey operations. This week the telescope logged {x_hours:.0f} hours of open-shutter time across {y_visits} science visits, with median delivered image quality of {z_fwhm:.2f}\" FWHM -- {vs_design} the design target.

[Brief note on any notable engineering event, e.g., dome/LLBV commissioning status, fault downtime, or a resolved issue.]

{footprint_sentence}{ddf_sentence}

{prompt_sentence}

Looking ahead, [1 sentence on upcoming priorities -- e.g., continued LLBV commissioning, image quality improvement work per SOTN-004, or scheduler tuning]. As always, feedback from the science community on what's most useful in these updates is welcome."""

display(Markdown(post))
print(post)